# Diseño de Muro de Concreto Reforzado
Cálculo conforme a la Norma Técnica Complementaria para Diseño y Construcción de Estructuras de Concreto (NTC 2023).

In [3]:
import math

def calcular_muro_concreto():
    print("=== DATOS DE ENTRADA ===")
    # Geometría
    Hm = float(input("Altura del muro (Hm) en cm: "))
    Lm = float(input("Longitud total del muro (Lm) en cm: "))
    tm = float(input("Espesor del muro (tm) en cm: "))
    hc2 = float(input("Longitud del elemento extremo izquierdo (hc2) en cm: "))
    bc2 = float(input("Espesor del elemento extremo izquierdo (bc2) en cm: "))
    hc1 = float(input("Longitud del elemento extremo derecho (hc1) en cm: "))
    bc1 = float(input("Espesor del elemento extremo derecho (bc1) en cm: "))
    
    # Materiales
    f_c = float(input("Resistencia del concreto (f'c) en kg/cm^2: "))
    f_y = float(input("Esfuerzo de fluencia del acero (fy) en kg/cm^2: "))
    clase_concreto = input("Clase de concreto (1A, 1B o 2): ").upper()
    
    # Cargas
    Pu = float(input("Carga axial última (Pu) en ton: ")) * 1000  # Convertir a kg
    Mu = float(input("Momento último (Mu) en ton*m: ")) * 100000  # Convertir a kg*cm
    Vu = float(input("Cortante último (Vu) en ton: ")) * 1000    # Convertir a kg

    print("\n=== PROPIEDADES DE LOS MATERIALES ===")
    # Cálculo de beta 1 (NTC 2023)
    if f_c <= 300:
        beta1 = 0.85
    else:
        beta1 = max(0.65, 1.05 - (f_c / 1400))
    print(f"Factor beta_1 = {beta1:.3f}")

    # Módulo de elasticidad (NTC 2023, Tabla 2.2.1)
    if clase_concreto == '1A':
        Ec = 14000 * math.sqrt(f_c)
    elif clase_concreto == '2':
        Ec = 8000 * math.sqrt(f_c)
    else:
        Ec = 11000 * math.sqrt(f_c) # Asumiendo promedio para 1B sin especificar agregado
    print(f"Módulo de elasticidad (Ec) = {Ec:.2f} kg/cm^2")
    
    # Esfuerzo equivalente
    f_bi = 0.85 * f_c

    print("\n=== DISEÑO POR RESISTENCIA (ESTADOS LÍMITE DE FALLA) ===")
    phi_flex = 0.90
    phi_cort = 0.75
    
    # Geometría para el método de T y C
    # Distancia entre centroides de los elementos extremos
    d1 = hc1 / 2
    d2 = Lm - (hc2 / 2)
    jd = d2 - d1 
    
    # CASO 1: Compresión en el extremo derecho
    print("\n--- CASO 1: Compresión en el extremo derecho ---")
    # Tensión en el extremo izquierdo y compresión en el derecho
    T_u1 = (Mu / jd) - (Pu / 2)
    C_u1 = (Mu / jd) + (Pu / 2)
    
    # Área de acero requerida en tensión (extremo izquierdo)
    As_req_izq = max(0, T_u1 / (phi_flex * f_y))
    
    # Profundidad del bloque de compresión (a) y eje neutro (c) en extremo derecho
    # C_u1 = phi_flex * f_bi * a * bc1 + As_req_der * f_y (Asumiendo que el concreto resiste la mayoría)
    a1 = C_u1 / (phi_flex * f_bi * bc1)
    if a1 > hc1:
        # Si el bloque supera el elemento extremo, requerimos acero de compresión adicional
        a1 = hc1
        C_concreto1 = phi_flex * f_bi * a1 * bc1
        C_exceso1 = C_u1 - C_concreto1
        As_req_der = C_exceso1 / (phi_flex * f_y)
    else:
        As_req_der = 0.0 # Porcentaje mínimo normativo regirá
        
    c1 = a1 / beta1

    # CASO 2: Compresión en el extremo izquierdo
    print("\n--- CASO 2: Compresión en el extremo izquierdo ---")
    T_u2 = (Mu / jd) - (Pu / 2)
    C_u2 = (Mu / jd) + (Pu / 2)
    
    As_req_der_t = max(0, T_u2 / (phi_flex * f_y))
    
    a2 = C_u2 / (phi_flex * f_bi * bc2)
    if a2 > hc2:
        a2 = hc2
        C_concreto2 = phi_flex * f_bi * a2 * bc2
        C_exceso2 = C_u2 - C_concreto2
        As_req_izq_c = C_exceso2 / (phi_flex * f_y)
    else:
        As_req_izq_c = 0.0
        
    c2 = a2 / beta1
    
    # Acero final en extremos (Máximo de ambos casos + cuantía mínima)
    rho_min = 0.002
    As_izq_final = max(As_req_izq, As_req_izq_c, rho_min * bc2 * hc2)
    As_der_final = max(As_req_der, As_req_der_t, rho_min * bc1 * hc1)

    print("\n--- REVISIÓN POR CORTANTE ---")
    # Resistencia del concreto al cortante
    V_cR = phi_cort * (0.5 * math.sqrt(f_c) * tm * d2)
    if Vu > V_cR:
        V_sR = Vu - V_cR
        # Separación horizontal y área requerida
        rho_h = V_sR / (phi_cort * f_y * tm * d2)
    else:
        rho_h = 0.0025 # Cuantía mínima NTC
        
    Ash_req = rho_h * tm # cm2/cm
    Asv_centro = 0.0025 * tm * (Lm - hc1 - hc2) # Cuantía mínima en el alma del muro

    print("\n=== RESULTADOS FINALES Y ESQUEMA ===")
    print(f"1. Área de acero vertical requerido en extremo izquierdo: {As_izq_final:.2f} cm^2")
    print(f"2. Área de acero vertical requerido en extremo derecho: {As_der_final:.2f} cm^2")
    print(f"3. Área de acero horizontal (extremos y alma): {Ash_req:.4f} cm^2/cm")
    print(f"4. Profundidad del eje neutro (c) - Caso 1: {c1:.2f} cm")
    print(f"5. Profundidad del eje neutro (c) - Caso 2: {c2:.2f} cm")
    print(f"6. Área de acero vertical al centro del muro (alma): {Asv_centro:.2f} cm^2")

# calcular_muro_concreto()

